# IDC 数据清洗、MOA 标注与关键图表

本版本已完成以下整理：

1. 删除重复、过时和会覆盖结果的代码；
2. 仅合并明确的格式重复类别；
3. 保留原始字段，并新增 `*_clean`、分组和审计标注字段；
4. 统一靶点名称，并对少量明确错误进行药物级修正；
5. 主要依据 `mechanism_of_action` 描述进行简洁 MOA 分类，必要时使用 `target_clean` 辅助；
6. 对错误或信息不足的 MOA 描述保留人工复核标记；
7. 输出论文所需的 ADA 分布图、target/MOA 分布图、ADA≥10% 比例及 Wilson 95% CI；
8. 保存清洗标注后的 Excel 表和人工复核表。

**运行顺序：从上到下执行全部单元格。不要再运行旧版本中的 `classify_moa` 或其他 `df["moa_group"] = ...` 代码。**

## 0. 导入、路径和数据读取

代码会优先寻找 `IDC_modeling_table_v0(1).xlsx`，也兼容不带 `(1)` 的文件名。

In [ ]:
from pathlib import Path
import re
import unicodedata
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ---------- 输入文件 ----------
DATA_CANDIDATES = [
    Path("IDC_modeling_table_cleaned_annotated.xlsx"),
    Path("IDC_modeling_table_v0.xlsx"),
]

DATA_PATH = next(
    (path for path in DATA_CANDIDATES if path.exists()),
    None
)

if DATA_PATH is None:
    raise FileNotFoundError(
        "未找到 IDC modeling table。请将 Excel 文件放在 Notebook 同目录，"
        "并命名为 IDC_modeling_table_v0(1).xlsx 或 IDC_modeling_table_v0.xlsx。"
    )


# ---------- 输出目录 ----------
OUTPUT_DIR = Path("IDC_cleaning_outputs")
MAIN_PLOT_DIR = OUTPUT_DIR / "plots" / "01_main"
SUPPLEMENT_PLOT_DIR = OUTPUT_DIR / "plots" / "02_supplementary"
REVIEW_DIR = OUTPUT_DIR / "manual_review"

for folder in [
    OUTPUT_DIR,
    MAIN_PLOT_DIR,
    SUPPLEMENT_PLOT_DIR,
    REVIEW_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)


# ---------- 读取并保留原始副本 ----------
df_raw = pd.read_excel(DATA_PATH)
df = df_raw.copy()

required_columns = {
    "molecule_inn_name",
    "therapeutic_id",
    "trial_id",
    "ada_frequency_percent",
    "target",
    "mechanism_of_action",
}

missing_columns = sorted(required_columns - set(df.columns))
if missing_columns:
    raise KeyError(f"缺少必要字段：{missing_columns}")


# ---------- 统一结局标签 ----------
df["ada_high_10_original"] = df.get("ada_high_10")

df["ada_frequency_percent"] = pd.to_numeric(
    df["ada_frequency_percent"],
    errors="coerce"
)

df["ada_high_10"] = pd.Series(
    pd.NA,
    index=df.index,
    dtype="Int64"
)

valid_ada = df["ada_frequency_percent"].notna()
df.loc[valid_ada, "ada_high_10"] = (
    df.loc[valid_ada, "ada_frequency_percent"] >= 10
).astype(int)

print("输入文件：", DATA_PATH.resolve())
print("数据维度：", df.shape)
print("\nADA≥10% 标签分布：")
display(df["ada_high_10"].value_counts(dropna=False))

In [ ]:
# ---------- 通用数据质量处理（可重复执行） ----------
# 1) 规范字段名；2) 清理标识符首尾空格；3) 删除完全重复行；
# 4) 检查主键冲突、必填缺失和数值范围；5) 保存数据质量审计。
df.columns = [str(c).strip() for c in df.columns]

identifier_columns = [
    "idc_row_id", "trial_id", "external_source_id",
    "therapeutic_id", "molecule_inn_name",
]
for column in identifier_columns:
    if column in df.columns:
        df[column] = df[column].astype("string").str.strip()
        df[column] = df[column].replace("", pd.NA)

rows_before_deduplication = len(df)
aggregate_start = (
    df.columns.get_loc("cohort_group_id")
    if "cohort_group_id" in df.columns
    else len(df.columns)
)
source_observation_columns = list(df.columns[:aggregate_start])
redundant_source_mask = df.duplicated(
    subset=source_observation_columns,
    keep="first",
)
excluded_duplicate_rows = df.loc[redundant_source_mask].copy()
excluded_duplicate_rows["exclusion_reason"] = (
    "Duplicate source observation; redundant copy differed only through stale/cross-matched aggregate fields."
)
redundant_source_rows_removed = int(redundant_source_mask.sum())
df = df.loc[~redundant_source_mask].copy()

if "idc_row_id" in df.columns:
    duplicate_id_mask = df["idc_row_id"].notna() & df["idc_row_id"].duplicated(keep=False)
else:
    duplicate_id_mask = pd.Series(False, index=df.index)

required_missing = {
    column: int(df[column].isna().sum())
    for column in required_columns
}

ada_out_of_range = int((
    df["ada_frequency_percent"].notna()
    & ~df["ada_frequency_percent"].between(0, 100)
).sum())

data_quality_audit = pd.DataFrame([
    {"check": "rows_before_deduplication", "value": rows_before_deduplication, "status": "INFO"},
    {"check": "redundant_source_rows_removed", "value": redundant_source_rows_removed, "status": "FIXED" if redundant_source_rows_removed else "PASS"},
    {"check": "remaining_duplicate_id_rows", "value": int(duplicate_id_mask.sum()), "status": "PENDING_FINAL_REKEY" if duplicate_id_mask.any() else "PASS"},
    {"check": "ada_frequency_out_of_range", "value": ada_out_of_range, "status": "REVIEW" if ada_out_of_range else "PASS"},
    *[
        {"check": f"missing_required::{column}", "value": count, "status": "REVIEW" if count else "PASS"}
        for column, count in sorted(required_missing.items())
    ],
])

duplicate_id_review = df.loc[duplicate_id_mask].copy()
data_quality_audit.to_excel(REVIEW_DIR / "data_quality_audit.xlsx", index=False)
duplicate_id_review.to_excel(REVIEW_DIR / "duplicate_id_review.xlsx", index=False)

assert ada_out_of_range == 0, "ada_frequency_percent 存在超出 0–100 的值，请复核。"
print(data_quality_audit.to_string(index=False))


## 1. ADA 结局的基础分布

这两张图用于论文的数据集描述：

- 连续 ADA frequency 的总体分布；
- 二分类结局 `ADA ≥10%` 的样本量。

In [ ]:
# ADA frequency 直方图
ada_values = df["ada_frequency_percent"].dropna()

plt.figure(figsize=(8, 5))
plt.hist(ada_values, bins=30)
plt.xlabel("ADA frequency (%)")
plt.ylabel("Number of records")
plt.title("Distribution of ADA frequency")
plt.tight_layout()
plt.savefig(
    MAIN_PLOT_DIR / "ada_frequency_distribution.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close()


# ADA ≥10% 类别数量
ada_class_counts = (
    df["ada_high_10"]
    .astype("string")
    .fillna("Missing")
    .replace({
        "0": "ADA <10%",
        "1": "ADA ≥10%"
    })
    .value_counts()
)

plt.figure(figsize=(6, 4))
plt.bar(ada_class_counts.index, ada_class_counts.values)
plt.ylabel("Number of records")
plt.title("Distribution of high ADA outcome")
plt.tight_layout()
plt.savefig(
    MAIN_PLOT_DIR / "ada_high10_distribution.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close()

## 2. 只清理明确重复的简单分类字段

这里不做无意义标准化，仅处理：

- Unicode、不可见空格、首尾空格；
- 大小写或格式上确定相同的类别；
- 明确拼写错误。

原始列全部保留，新列使用 `_clean` 后缀。

In [ ]:
def clean_text(value):
    """清理 Unicode、不可见空格和多余空格，不改变生物学含义。"""
    if pd.isna(value):
        return pd.NA

    text = unicodedata.normalize("NFKC", str(value))
    text = text.replace("\xa0", " ")
    text = re.sub(r"[‐-‒–—−]", "-", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text if text else pd.NA


def map_by_clean_key(series, mapping):
    """
    仅替换 mapping 中明确列出的类别。
    匹配时忽略大小写和首尾空格，未列入的值保持原样。
    """
    clean_mapping = {
        clean_text(key).casefold(): value
        for key, value in mapping.items()
    }

    def convert(value):
        text = clean_text(value)

        if pd.isna(text):
            return pd.NA

        return clean_mapping.get(
            text.casefold(),
            text
        )

    return series.apply(convert)


df["light_chain_clean"] = map_by_clean_key(
    df["light_chain"],
    {
        "kappa": "Kappa",
        "lambda": "Lambda",
        "not applicable": "Not applicable",
    }
)

df["antibody_backbone_clean"] = map_by_clean_key(
    df["antibody_backbone"],
    {
        "human IgG1": "Human IgG1",
        "human IgG2": "Human IgG2",
        "human IgG4": "Human IgG4",
        "scfv": "scFv",
        "not applicable": "Not applicable",
    }
)

df["conjugate_modification_clean"] = map_by_clean_key(
    df["conjugate_modification"],
    {
        "not applicable": "Not applicable",
        "unconjugated": "Unconjugated",
        "PEG conjugate": "PEG conjugate",
        "drug conjugate": "Drug conjugate",
        "radioisotope conjugate": "Radioisotope conjugate",
    }
)

df["fc_modifications_clean"] = map_by_clean_key(
    df["fc_modifications"],
    {
        "not applicable": "Not applicable",
    }
)

df["disease_category_clean"] = map_by_clean_key(
    df["disease_indication_category"],
    {
        "Metobolic and endocrine disorders":
            "Metabolic and endocrine disorders",
        "Blood Disorders":
            "Blood disorders",
    }
)

# 当前 route 本身已经较干净，仅做文本空格清理
df["route_clean"] = df["route"].apply(clean_text)

In [ ]:
# 合并前后类别数审计
simple_cleaning_audit = pd.DataFrame([
    {
        "original_column": "light_chain",
        "clean_column": "light_chain_clean",
        "unique_before": df["light_chain"].nunique(dropna=False),
        "unique_after": df["light_chain_clean"].nunique(dropna=False),
    },
    {
        "original_column": "antibody_backbone",
        "clean_column": "antibody_backbone_clean",
        "unique_before": df["antibody_backbone"].nunique(dropna=False),
        "unique_after": df["antibody_backbone_clean"].nunique(dropna=False),
    },
    {
        "original_column": "conjugate_modification",
        "clean_column": "conjugate_modification_clean",
        "unique_before": df["conjugate_modification"].nunique(dropna=False),
        "unique_after": df["conjugate_modification_clean"].nunique(dropna=False),
    },
    {
        "original_column": "fc_modifications",
        "clean_column": "fc_modifications_clean",
        "unique_before": df["fc_modifications"].nunique(dropna=False),
        "unique_after": df["fc_modifications_clean"].nunique(dropna=False),
    },
])

display(simple_cleaning_audit)

simple_cleaning_audit.to_excel(
    REVIEW_DIR / "simple_category_cleaning_audit.xlsx",
    index=False
)

## 3. 靶点名称清洗与必要修正

处理原则：

- 标准缩写、希腊字母和已确认同义词；
- 多靶点仍保留为多靶点；
- 仅对已确认的药物级错误进行修正；
- 使用审计字段记录修正原因。

In [ ]:
TARGET_NAME_MAP = {
    "TNFa": "TNF-α",
    "TNF alpha": "TNF-α",
    "CD274 (PDL1)": "PD-L1",
    "PDL1": "PD-L1",
    "PDCD1 (PD-1)": "PD-1",
    "PDCD1": "PD-1",
    "IL-6R": "IL-6 receptor",
    "IL6R": "IL-6 receptor",
    "IL-4Ra": "IL-4 receptor α",
    "IL-5Ra": "IL-5 receptor α",
    "IL-17RA": "IL-17 receptor A",
    "IL-1ß": "IL-1β",
    "VEGFR2 (KDR)": "VEGFR2",
    "RANK-L": "RANKL",
    "Integrin-α4β7": "α4β7 integrin",
    "Alpa4beta1 integrin": "α4β1 integrin",
    "PDGFR-α": "PDGFRα",
    "4-1BB": "4-1BB/CD137",
    "BLyS (BAFF)": "BAFF",
    "RSV F Protein": "RSV F protein",
    "RSV gpF": "RSV F protein",
    "INFγ": "IFN-γ",
    "CD3e": "CD3ε",
    "hyaluronan": "Hyaluronan",
    "CD25 (IL-2R)": "CD25/IL-2Rα",
    "HER2, tubulin": "HER2 + tubulin",
    "CD19, CD3": "CD19 + CD3",
    "IL-12, IL-23": "IL-12 + IL-23",
    "FVIII, FX": "Factor VIII + Factor X",
    "C. difficile toxin B":
        "Clostridioides difficile toxin B",
    "IL-17A, IL-17F": "IL-17A + IL-17F",
    "IL-6R, Albumin": "IL-6 receptor + albumin",
    "Ang1/2": "Angiopoietin-1/2",
    "Glycosaminoglycan": "Glycosaminoglycans",
    "VEGF, Ang2": "VEGF-A + Angiopoietin-2",
    "globotriaosylceramide (GL-3)":
        "Globotriaosylceramide (Gb3/GL-3)",
}

df["target_clean"] = map_by_clean_key(
    df["target"],
    TARGET_NAME_MAP
)

df["target_correction_flag"] = False
df["target_correction_note"] = pd.NA
df["half_life_extension_partner"] = pd.NA

drug_name = (
    df["molecule_inn_name"]
    .astype("string")
    .str.strip()
)


# 药物身份纠错：该记录的 therapeutic_id=PR_0454、trade name=RG-7593、
# CD22/MMAE 机制均与 Pinatuzumab Vedotin 一致，而非 Bevacizumab。
df["molecule_inn_name_original"] = df["molecule_inn_name"]
mislabelled_pinatuzumab = (
    drug_name.eq("Bevacizumab")
    & df["target_clean"].eq("CD22")
    & df["mechanism_of_action"].astype("string").str.contains("MMAE", case=False, na=False)
)
df.loc[mislabelled_pinatuzumab, "molecule_inn_name"] = "Pinatuzumab Vedotin"
df.loc[mislabelled_pinatuzumab, "target_correction_flag"] = True
df.loc[mislabelled_pinatuzumab, "target_correction_note"] = (
    "Corrected molecule identity from Bevacizumab to Pinatuzumab Vedotin; "
    "PR_0454/RG-7593 and CD22-directed MMAE payload are consistent with pinatuzumab vedotin."
)
drug_name = df["molecule_inn_name"].astype("string").str.strip()

assert not (
    drug_name.eq("Bevacizumab") & df["target_clean"].eq("CD22")
).any(), "仍存在 Bevacizumab–CD22 错误映射。"


def apply_target_correction(molecules, corrected_target, note):
    mask = drug_name.isin(set(molecules))

    df.loc[mask, "target_clean"] = corrected_target
    df.loc[mask, "target_correction_flag"] = True
    df.loc[mask, "target_correction_note"] = note


apply_target_correction(
    {"Galcanezumab", "Eptinezumab", "Fremanezumab"},
    "CGRP ligand",
    "Corrected from CGRP receptor to CGRP ligand based on molecule identity."
)

apply_target_correction(
    {"Erenumab"},
    "CGRP receptor",
    "Confirmed as CGRP receptor-targeting antibody."
)

apply_target_correction(
    {"Nonacog alfa", "Recombinant factor IX (rFIX)"},
    "Coagulation factor IX",
    "Original target field described downstream coagulation factors; recoded as factor IX replacement product."
)

apply_target_correction(
    {"Vobarilizumab"},
    "IL-6 receptor",
    "Albumin binding is a half-life-extension feature, not the primary pharmacological target."
)

df.loc[
    drug_name.eq("Vobarilizumab"),
    "half_life_extension_partner"
] = "Albumin"

In [ ]:
TARGET_GROUP_MAP = {
    # 免疫检查点与共刺激
    "PD-1": "Immune checkpoint/co-stimulation",
    "PD-L1": "Immune checkpoint/co-stimulation",
    "CTLA-4": "Immune checkpoint/co-stimulation",
    "OX40": "Immune checkpoint/co-stimulation",
    "4-1BB/CD137": "Immune checkpoint/co-stimulation",
    "CD47": "Immune checkpoint/co-stimulation",
    "NKG2A": "Immune checkpoint/co-stimulation",
    "CD73": "Immune checkpoint/co-stimulation",

    # 免疫细胞表面抗原
    "CD20": "Immune-cell surface antigen",
    "CD22": "Immune-cell surface antigen",
    "CD30 (TNFRSF8)": "Immune-cell surface antigen",
    "CD33": "Immune-cell surface antigen",
    "CD38": "Immune-cell surface antigen",
    "CD52": "Immune-cell surface antigen",
    "CD79b": "Immune-cell surface antigen",
    "SLAMF7": "Immune-cell surface antigen",
    "CD3": "Immune-cell surface antigen",
    "CD3ε": "Immune-cell surface antigen",

    # 细胞因子与免疫介质
    "TNF-α": "Cytokine/immune mediator",
    "IL-17A": "Cytokine/immune mediator",
    "IL-13": "Cytokine/immune mediator",
    "IL-5": "Cytokine/immune mediator",
    "IL-1β": "Cytokine/immune mediator",
    "IL-1α": "Cytokine/immune mediator",
    "IL-12 + IL-23": "Cytokine/immune mediator",
    "IL-17A + IL-17F": "Cytokine/immune mediator",
    "IL-33": "Cytokine/immune mediator",
    "IFN-γ": "Cytokine/immune mediator",
    "IL-6": "Cytokine/immune mediator",
    "MIF": "Cytokine/immune mediator",
    "BAFF": "Cytokine/immune mediator",
    "IgE": "Cytokine/immune mediator",

    # 细胞因子受体
    "IL-6 receptor": "Cytokine receptor",
    "IL-4 receptor α": "Cytokine receptor",
    "IL-5 receptor α": "Cytokine receptor",
    "IL-17 receptor A": "Cytokine receptor",
    "CD25/IL-2Rα": "Cytokine receptor",
    "CSF1R/CD115": "Cytokine receptor",
    "IL-21R": "Cytokine receptor",

    # 神经营养与神经肽
    "NGF": "Neurotrophic/neuropeptide pathway",
    "CGRP receptor": "Neurotrophic/neuropeptide pathway",
    "CGRP ligand": "Neurotrophic/neuropeptide pathway",

    # 生长因子与肿瘤信号
    "HER2": "Growth factor/oncogenic pathway",
    "EGFR": "Growth factor/oncogenic pathway",
    "VEGFR2": "Growth factor/oncogenic pathway",
    "PDGFRα": "Growth factor/oncogenic pathway",
    "MET receptor": "Growth factor/oncogenic pathway",
    "VEGF-A": "Growth factor/oncogenic pathway",
    "VEGF": "Growth factor/oncogenic pathway",

    # 代谢
    "PCSK9": "Metabolic/lipid regulator",
    "Phenylalanine": "Metabolic substrate",
    "L-asparagine": "Metabolic substrate",
    "Glycogen": "Metabolic substrate",
    "Glycosaminoglycans": "Metabolic substrate",
    "Globotriaosylceramide (Gb3/GL-3)": "Metabolic substrate",

    # 骨与矿物代谢
    "RANKL": "Bone/mineral metabolism",
    "Sclerostin": "Bone/mineral metabolism",
    "FGF23": "Bone/mineral metabolism",

    # 黏附与迁移
    "α4β7 integrin": "Adhesion/trafficking",
    "α4β1 integrin": "Adhesion/trafficking",
    "β7 integrin": "Adhesion/trafficking",
    "CD11a": "Adhesion/trafficking",

    # 补体、凝血与蛋白酶
    "C5": "Complement/coagulation/protease",
    "Plasma Kallikrein": "Complement/coagulation/protease",
    "Factor VIII + Factor X": "Complement/coagulation/protease",
    "Coagulation factor IX": "Complement/coagulation/protease",

    # 病原体与毒素
    "RSV F protein": "Pathogen/toxin antigen",
    "Protective antigen of Bacillus anthracis":
        "Pathogen/toxin antigen",
    "Clostridioides difficile toxin B":
        "Pathogen/toxin antigen",

    # 基质与递送
    "Hyaluronan": "Extracellular matrix/drug delivery",

    # 多靶点
    "HER2 + tubulin": "Multi-target/composite",
    "CD19 + CD3": "Multi-target/composite",
    "IL-6 receptor + albumin": "Multi-target/composite",
    "VEGF-A + Angiopoietin-2": "Multi-target/composite",

    # 血管生成
    "Angiopoietin-1/2": "Angiogenesis pathway",

    # 药物逆转
    "Dabigatran": "Drug neutralization/reversal",
}

df["target_group"] = (
    df["target_clean"]
    .map(TARGET_GROUP_MAP)
    .fillna("Other target")
)

remaining_targets = (
    df.loc[
        df["target_group"].eq("Other target"),
        "target_clean"
    ]
    .value_counts(dropna=False)
    .rename_axis("target_clean")
    .reset_index(name="row_count")
)

display(remaining_targets)

remaining_targets.to_excel(
    REVIEW_DIR / "remaining_targets.xlsx",
    index=False
)

## 4. MOA 清洗和简洁分类

分类原则：

- 先识别替代治疗、酶抑制、补体抑制、ADC/毒素载荷、T-cell engager 等明确机制；
- 再依据 `target_clean` 区分配体中和、受体阻断、免疫检查点阻断和黏附迁移阻断；
- 对原始描述明显错误或信息不足的少量药物，使用人工复核覆盖；
- 保留 `mechanism_of_action` 原文，并新增 `mechanism_of_action_reviewed`、来源和复核标记。

In [ ]:
def normalize_moa_key(value):
    """用于 MOA 文本和 target 的稳健匹配。"""
    if pd.isna(value):
        return ""

    text = unicodedata.normalize("NFKC", str(value))
    text = re.sub(r"[‐-‒–—−]", "-", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text.casefold()


CHECKPOINT_TARGETS = {
    "PD-1", "PD-L1", "CTLA-4", "CD47", "NKG2A", "CD73"
}

ADHESION_TARGETS = {
    "α4β7 integrin", "α4β1 integrin", "β7 integrin", "CD11a"
}

RECEPTOR_TARGETS = {
    "CGRP receptor",
    "VEGFR2",
    "PDGFRα",
    "EGFR",
    "HER2",
    "MET receptor",
    "IL-6 receptor",
    "IL-4 receptor α",
    "IL-5 receptor α",
    "IL-17 receptor A",
    "IL-21R",
    "CSF1R/CD115",
    "CD25/IL-2Rα",
}

LIGAND_TARGETS = {
    "NGF",
    "PCSK9",
    "CGRP ligand",
    "RANKL",
    "VEGF-A",
    "VEGF",
    "VEGF-A + Angiopoietin-2",
    "IL-1β",
    "IL-1α",
    "IL-5",
    "IL-6",
    "IL-13",
    "IL-17A",
    "IL-17A + IL-17F",
    "IL-12 + IL-23",
    "IL-33",
    "IFN-γ",
    "MIF",
    "IgE",
    "TNF-α",
    "BAFF",
    "Angiopoietin-1/2",
    "Sclerostin",
    "FGF23",
}

IMMUNE_CELL_TARGETS = {
    "CD3",
    "CD3ε",
    "CD20",
    "CD22",
    "CD30 (TNFRSF8)",
    "CD33",
    "CD38",
    "CD52",
    "CD79b",
    "SLAMF7",
}

PATHOGEN_TARGETS = {
    "RSV F protein",
    "Protective antigen of Bacillus anthracis",
    "Clostridioides difficile toxin B",
}


def normalized_set(values):
    return {normalize_moa_key(value) for value in values}


CHECKPOINT_KEYS = normalized_set(CHECKPOINT_TARGETS)
ADHESION_KEYS = normalized_set(ADHESION_TARGETS)
RECEPTOR_KEYS = normalized_set(RECEPTOR_TARGETS)
LIGAND_KEYS = normalized_set(LIGAND_TARGETS)
IMMUNE_CELL_KEYS = normalized_set(IMMUNE_CELL_TARGETS)
PATHOGEN_KEYS = normalized_set(PATHOGEN_TARGETS)

In [ ]:
# 原始描述明显错误或过于简略的少量记录：
# 保留原文，并给出 reviewed 描述和人工复核原因。
MOA_REVIEW_CORRECTIONS = {
    "Faricimab": {
        "reviewed_description":
            "Bispecific antibody that simultaneously inhibits VEGF-A and angiopoietin-2, suppressing two complementary angiogenic pathways.",
        "moa_group": "Ligand neutralization",
        "note": "Completed a truncated MOA description ending in by.",
    },
    "Trebananib": {
        "reviewed_description":
            "Peptibody that binds angiopoietin-1 and angiopoietin-2 and prevents their interaction with the Tie2 receptor, thereby inhibiting angiogenesis.",
        "moa_group": "Ligand neutralization",
        "note": "Expanded the nonspecific original description antiangiogenic.",
    },
    "Cetuximab": {
        "reviewed_description":
            "Binds EGFR and blocks ligand binding and downstream receptor signaling.",
        "moa_group": "Receptor blockade/antagonism",
        "note": "Antagonist text previously matched the substring agonist.",
    },
    "Necitumumab": {
        "reviewed_description":
            "Binds EGFR and blocks ligand binding and downstream receptor signaling.",
        "moa_group": "Receptor blockade/antagonism",
        "note": "Antagonist text previously matched the substring agonist.",
    },
    "Reslizumab": {
        "reviewed_description":
            "Binds and neutralizes IL-5, preventing IL-5 receptor engagement and eosinophil signaling.",
        "moa_group": "Ligand neutralization",
        "note": "FDA labeling calls reslizumab an IL-5 antagonist, but the molecular target is the IL-5 ligand rather than its receptor.",
    },
    "Isatuximab": {
        "reviewed_description":
            "Targets CD38-positive cells through ADCC, ADCP, complement-dependent cytotoxicity, and direct apoptosis, and also inhibits CD38 ectoenzymatic activity.",
        "moa_group": "Immune-cell targeting/modulation",
        "note": "Reclassified from complement inhibition to reflect the dominant multi-mechanistic CD38-directed activity.",
    },
    "Moxetumomab pasudotox": {
        "reviewed_description":
            "CD22-directed immunotoxin delivering Pseudomonas exotoxin A to CD22-positive cells.",
        "moa_group": "Targeted payload delivery",
        "note": "Original MOA text was only 'Targeted' and was insufficient for classification.",
    },
    "Bimekizumab": {
        "reviewed_description":
            "Neutralizes IL-17A and IL-17F and prevents signaling through the IL-17 receptor complex.",
        "moa_group": "Ligand neutralization",
        "note": "Original MOA text was inconsistent with the recorded target.",
    },
    "Itepekimab": {
        "reviewed_description":
            "Neutralizes IL-33 and inhibits IL-33-mediated signaling.",
        "moa_group": "Ligand neutralization",
        "note": "Original MOA text incorrectly described VEGF-A and Ang-2.",
    },
    "Oleclumab": {
        "reviewed_description":
            "Inhibits CD73-mediated extracellular adenosine production and adenosine-dependent immune suppression.",
        "moa_group": "Immune checkpoint blockade",
        "note": "Original MOA text was truncated and inconsistent with the recorded CD73 target.",
    },
}

df["mechanism_of_action_reviewed"] = df["mechanism_of_action"].apply(
    clean_text
)
df["moa_review_flag"] = False
df["moa_review_note"] = pd.NA

for molecule, correction in MOA_REVIEW_CORRECTIONS.items():
    mask = drug_name.eq(molecule)

    df.loc[
        mask,
        "mechanism_of_action_reviewed"
    ] = correction["reviewed_description"]

    df.loc[mask, "moa_review_flag"] = True
    df.loc[mask, "moa_review_note"] = correction["note"]

In [ ]:
def classify_moa_final(row):
    """
    主要根据 reviewed MOA 描述分类，target_clean 仅作为必要辅助。
    返回：moa_group, moa_classification_source
    """
    molecule = clean_text(row.get("molecule_inn_name"))
    moa_text = normalize_moa_key(
        row.get("mechanism_of_action_reviewed")
    )
    target_key = normalize_moa_key(
        row.get("target_clean")
    )

    # 先应用少量经过人工核查的覆盖
    if molecule in MOA_REVIEW_CORRECTIONS:
        return pd.Series([
            MOA_REVIEW_CORRECTIONS[molecule]["moa_group"],
            "Manual review",
        ])

    if not moa_text:
        return pd.Series([
            "Missing",
            "Missing MOA description",
        ])

    # 1. 药物中和或逆转
    if any(term in moa_text for term in [
        "reversal agent",
        "reverse the anticoagulant",
        "dabigatran",
    ]):
        return pd.Series([
            "Drug neutralization/reversal",
            "MOA description",
        ])

    # 2. 组织扩散或给药辅助
    if any(term in moa_text for term in [
        "hyaluronidase",
        "dispersion and absorption",
        "degrades hyaluronan",
        "tissue dispersion",
    ]):
        return pd.Series([
            "Tissue dispersion/drug delivery",
            "MOA description",
        ])

    # 3. 靶向载荷递送
    if (
        target_key == normalize_moa_key("HER2 + tubulin")
        or any(term in moa_text for term in [
            "antibody drug conjugate",
            "antibody-drug conjugate",
            "drug conjugate",
            "cytotoxic payload",
            "payload delivery",
            "mmae",
            "calicheamicin",
            "pseudomonas exotoxin",
            "diphtheria toxin",
            "immunotoxin",
            "microtubule inhibitor",
        ])
    ):
        return pd.Series([
            "Targeted payload delivery",
            "MOA description/target",
        ])

    # 4. 酶替代或底物降解
    if any(term in moa_text for term in [
        "enzyme replacement",
        "replacement enzyme",
        "exogenous enzyme",
        "cleaves the terminal",
        "cleaves glycogen",
        "degrades glycogen",
        "converts phenylalanine",
        "l-asparaginase",
        "galactosidase",
        "iduronate-2-sulfatase",
    ]):
        return pd.Series([
            "Enzyme replacement/substrate degradation",
            "MOA description",
        ])

    # 5. 凝血因子替代
    if any(term in moa_text for term in [
        "restores clotting",
        "clotting factor replacement",
        "factor ix replacement",
        "activation of factor x",
        "activation of fx",
    ]):
        return pd.Series([
            "Coagulation factor replacement",
            "MOA description",
        ])

    # 6. 补体抑制
    if "complement" in moa_text:
        return pd.Series([
            "Complement inhibition",
            "MOA description",
        ])

    # 7. 病原体或毒素中和
    if (
        target_key in PATHOGEN_KEYS
        or any(term in moa_text for term in [
            "respiratory syncytial virus",
            "rsv f protein",
            "block membrane fusion",
            "anthrax",
            "bacillus anthracis",
            "toxin b",
        ])
    ):
        return pd.Series([
            "Pathogen/toxin neutralization",
            "MOA description/target",
        ])

    # 8. T-cell engager
    if (
        "cd3" in moa_text
        and any(term in moa_text for term in [
            "cross linking",
            "cross-linking",
            "engager",
            "redirect",
            "recruit t cells",
        ])
    ):
        return pd.Series([
            "T-cell engager",
            "MOA description",
        ])

    # 9. 明确拮抗/阻断：必须先于 agonist 判断，避免 antagonist 包含 agonist 的子串误判。
    if any(term in moa_text for term in [
        "antagonist",
        "blocks ligand binding",
        "block ligand binding",
        "prevents binding of its ligands",
    ]):
        return pd.Series([
            "Receptor blockade/antagonism",
            "MOA description",
        ])

    # 9. 受体激动
    if any(term in moa_text for term in [
        "agonist",
        "agonistic",
        "t lymphocyte stimulants",
        "activates ox40",
        "cd137 antigen agonists",
        "cd137 agonist",
        "4-1bb agonist",
    ]):
        return pd.Series([
            "Receptor agonism",
            "MOA description",
        ])

    # 10. 免疫检查点阻断
    if target_key in CHECKPOINT_KEYS:
        return pd.Series([
            "Immune checkpoint blockade",
            "MOA description + target",
        ])

    # 11. 黏附或细胞迁移阻断
    if target_key in ADHESION_KEYS:
        return pd.Series([
            "Adhesion/trafficking blockade",
            "MOA description + target",
        ])

    # 12. 酶抑制
    if (
        target_key == normalize_moa_key("Plasma Kallikrein")
        or any(term in moa_text for term in [
            "plasma kallikrein",
            "enzyme inhibitor",
            "enzymatic activity",
        ])
    ):
        return pd.Series([
            "Enzyme inhibition",
            "MOA description/target",
        ])

    # 13. 明确的受体阻断
    if target_key in RECEPTOR_KEYS:
        return pd.Series([
            "Receptor blockade/antagonism",
            "MOA description + target",
        ])

    # 14. 明确的配体中和
    if target_key in LIGAND_KEYS:
        return pd.Series([
            "Ligand neutralization",
            "MOA description + target",
        ])

    # 15. 免疫细胞靶向或调节
    if (
        target_key in IMMUNE_CELL_KEYS
        or any(term in moa_text for term in [
            "depletion",
            "depletor",
            "adcc",
            "adcp",
            "cdc",
            "cytotoxicity",
            "autoreactive t cells",
            "immunostimulatory antibody",
            "against cd38",
            "anti-slamf7",
        ])
    ):
        return pd.Series([
            "Immune-cell targeting/modulation",
            "MOA description/target",
        ])

    return pd.Series([
        "Other/unclear",
        "Insufficient information",
    ])


# 本 Notebook 中唯一一次 moa_group 赋值
df[
    [
        "moa_group",
        "moa_classification_source",
    ]
] = df.apply(
    classify_moa_final,
    axis=1
)

In [ ]:
# ---------- 最终人工审核决定与建模前处理 ----------
# 保留原始标识符，修复来源、亚组主键和记录依赖；模型划分必须按 model_split_group 分组。
df["idc_row_id_original"] = df["idc_row_id"]
df["trial_id_original"] = df["trial_id"]
df["external_source_id_original"] = df["external_source_id"]
df["record_review_status"] = "Not individually reviewed"
df["record_review_note"] = pd.NA
df["record_dependency_note"] = pd.NA

def approve(mask, note):
    df.loc[mask, "record_review_status"] = "Approved"
    df.loc[mask, "record_review_note"] = note

# CT0736：补全 Maini 等研究 DOI；6个剂量×MTX亚组均为真实观察。
mask = df["trial_id"].eq("CT0736")
df.loc[mask, "external_source_id"] = "10.1002/1529-0131(199809)41:9<1552::AID-ART5>3.0.CO;2-W"
approve(mask, "Six dose-by-methotrexate strata retained and uniquely keyed.")

# CT0737：原文的合并免疫抑制包括 AZA、6-MP 或 MTX，不应只写 MTX。
mask = df["trial_id"].eq("CT0737") & df["coadministered_drugs"].notna()
df.loc[mask, "coadministered_drugs"] = "Immunosuppressive therapy"
approve(df["trial_id"].eq("CT0737"), "Two mutually exclusive immunosuppressive-therapy strata retained.")

# CT0745：12%/38% 为同一原始研究的 MTX/无MTX亚组；恢复原始论文 DOI。
mask = df["trial_id"].eq("CT0745") & df["coadministered_drugs"].notna()
df.loc[mask, "external_source_id"] = "10.1136/ard.2006.065615"
df.loc[mask, "coadministered_drugs"] = "Methotrexate"
approve(df["trial_id"].eq("CT0745"), "Two mutually exclusive methotrexate strata retained; primary DOI restored.")

# CT0751：一行 trial_id 误为 CT0685；两行是同一论文中的免疫调节剂亚组。
mask = df["idc_row_id"].eq("CT0751_A1_001")
df.loc[mask, "trial_id"] = "CT0751"
df.loc[mask & df["coadministered_drugs"].notna(), "coadministered_drugs"] = "Immunomodulator therapy"
approve(mask, "Trial ID harmonized to CT0751; two mutually exclusive strata retained.")

# CT0752：392天记录来自 CLASSIC II 维持期论文，不是 CLASSIC I 诱导期论文。
mask = df["idc_row_id"].eq("CT0752_A1_001") & df["assessment_days"].eq(392)
df.loc[mask, "external_source_id"] = "10.1136/gut.2006.106781"
df.loc[mask, "coadministered_drugs"] = "Immunosuppressive therapy"
df.loc[mask, "patient_population"] = "CLASSIC II maintenance cohort; 84 participants received concomitant immunosuppressive therapy."
df.loc[mask, "dosing_description"] = "Adalimumab maintenance 40 mg every other week or weekly through week 56 (CLASSIC II)."
approve(df["idc_row_id"].eq("CT0752_A1_001"), "CLASSIC I/II observations retained under one dependency group.")
df.loc[df["idc_row_id"].eq("CT0752_A1_001"), "record_dependency_note"] = "Longitudinally related CLASSIC I/II observations; keep in the same model split group."

# CT0742：总体队列与用药亚组存在嵌套，必须做聚类/分组分析或敏感性分析。
mask = df["idc_row_id"].eq("CT0742_A1_001")
approve(mask, "Overall cohort and nested concomitant-medication subgroup retained with dependency warning.")
df.loc[mask, "record_dependency_note"] = "Nested subgroup and overall cohort overlap; use grouped/clustered analysis or select one sensitivity-analysis row."

# 为真实的多亚组记录生成唯一主键。
def set_unique_row(mask, new_row_id, new_trial_id=None):
    assert int(mask.sum()) == 1, (new_row_id, int(mask.sum()))
    df.loc[mask, "idc_row_id"] = new_row_id
    if new_trial_id is not None:
        df.loc[mask, "trial_id"] = new_trial_id

mask = df["idc_row_id"].eq("CT0725_A1_001")
set_unique_row(mask & drug_name.eq("Adalimumab"), "CT0725_A1_001", "CT0725")
set_unique_row(mask & drug_name.eq("Etanercept"), "CT0725B_A1_001", "CT0725B")
approve(df["trial_id"].isin(["CT0725", "CT0725B"]), "Two independent publications separated into distinct study IDs.")

for arm in ("A1", "A2", "A3"):
    mask = df["idc_row_id"].eq(f"CT0736_{arm}_001")
    set_unique_row(mask & df["coadministered_drugs"].notna(), f"CT0736_{arm}_001")
    set_unique_row(mask & df["coadministered_drugs"].isna(), f"CT0736_{arm}_002")

for trial in ("CT0737", "CT0742", "CT0745", "CT0751"):
    mask = df["idc_row_id"].eq(f"{trial}_A1_001")
    set_unique_row(mask & df["coadministered_drugs"].notna(), f"{trial}_A1_001")
    set_unique_row(mask & df["coadministered_drugs"].isna(), f"{trial}_A1_002")

mask = df["idc_row_id"].eq("CT0752_A1_001")
set_unique_row(mask & df["assessment_days"].eq(28), "CT0752_A1_001")
set_unique_row(mask & df["assessment_days"].eq(392), "CT0752_A1_002")

df["has_coadministered_drugs"] = df["coadministered_drugs"].notna().astype(int)
df["comedication_missing"] = df["coadministered_drugs"].isna().astype(int)
df["ada_fraction"] = df["ada_frequency_percent"] / 100
df["ada_high_10"] = (df["ada_frequency_percent"] >= 10).astype("Int64")

count_available = df["n_ada_assessed"].notna() & df["n_ada_positive"].notna() & df["n_ada_assessed"].gt(0)
count_rate = df["n_ada_positive"] / df["n_ada_assessed"] * 100
count_delta = (count_rate - df["ada_frequency_percent"]).abs()
df["ada_count_consistency_status"] = np.select(
    [~count_available, count_delta.le(1.0)],
    ["COUNT_DATA_MISSING", "PASS"],
    default="REVIEW_COUNT_MISMATCH",
)
df["ada_count_rate_percent"] = count_rate
df["ada_count_delta_pp"] = count_delta
df["binomial_count_model_eligible"] = count_available & count_delta.le(1.0)
df["model_split_group"] = df["trial_id"]
df["modeling_include"] = True

required_model_fields = [
    "idc_row_id", "trial_id", "therapeutic_id", "molecule_inn_name",
    "ada_frequency_percent", "target_clean", "moa_group",
]
df["modeling_ready"] = df[required_model_fields].notna().all(axis=1)

TARGET_LEAKAGE_COLUMNS = [
    "ada_fraction", "n_ada_positive", "cohort_group_id", "cohort_ADA", "max_ADA_time", "N_at_max_ADA",
    "trial_group_id", "trial_ADA", "N_of_trial_ADA", "INN_group_id", "INN_ADA", "N_of_INN_ADA",
    "PR_group_id", "PRID_ADA", "N_of_PRID_ADA", "ada_high_10_original",
]
modeling_df = df.drop(columns=[c for c in TARGET_LEAKAGE_COLUMNS if c in df.columns]).copy()

assert len(modeling_df) == 2611
assert modeling_df["idc_row_id"].is_unique
assert modeling_df[required_model_fields].notna().all().all()
assert modeling_df["ada_frequency_percent"].between(0, 100).all()
assert not (modeling_df["molecule_inn_name"].eq("Bevacizumab") & modeling_df["target_clean"].eq("CD22")).any()
assert not modeling_df["target_group"].eq("Other target").any()
assert not modeling_df["moa_group"].isin(["Other/unclear", "Missing"]).any()

print("最终建模行数：", len(modeling_df))
print("唯一主键：", modeling_df["idc_row_id"].is_unique)
print("可用于二项计数模型的行数：", int(modeling_df["binomial_count_model_eligible"].sum()))


In [ ]:
# 输出仍需复核的 MOA，同时检查是否还有高频未分类记录
remaining_moa = (
    df.loc[
        df["moa_group"].isin([
            "Other/unclear",
            "Missing",
        ]),
        [
            "molecule_inn_name",
            "target_clean",
            "mechanism_of_action",
            "mechanism_of_action_reviewed",
            "moa_group",
        ]
    ]
    .groupby(
        [
            "molecule_inn_name",
            "target_clean",
            "mechanism_of_action",
            "mechanism_of_action_reviewed",
            "moa_group",
        ],
        dropna=False
    )
    .size()
    .reset_index(name="row_count")
    .sort_values("row_count", ascending=False)
)

display(remaining_moa)

remaining_moa.to_excel(
    REVIEW_DIR / "remaining_moa_review.xlsx",
    index=False
)

manual_moa_review = (
    df.loc[
        df["moa_review_flag"],
        [
            "molecule_inn_name",
            "target",
            "target_clean",
            "mechanism_of_action",
            "mechanism_of_action_reviewed",
            "moa_group",
            "moa_review_note",
        ]
    ]
    .drop_duplicates()
)

display(manual_moa_review)

manual_moa_review.to_excel(
    REVIEW_DIR / "manual_moa_corrections.xlsx",
    index=False
)

high_frequency_unclassified = remaining_moa.loc[
    remaining_moa["row_count"] >= 10
]

assert high_frequency_unclassified.empty, (
    "仍有至少10条记录的高频MOA未分类，请先人工检查：\n"
    + high_frequency_unclassified.to_string(index=False)
)

## 5. ADA assay 平台提取

只提取检测平台，不对原始长文本做过度标准化。

In [ ]:
def classify_assay_platform(value):
    if pd.isna(value):
        return "Not reported"

    text = clean_text(value).casefold()

    missing_tokens = {
        "not reported",
        "not available",
        "not provided",
        "unknown",
        "unspecified",
        "nr",
        "n/a"
    }

    if text in missing_tokens:
        return "Not reported"

    # 后面继续原来的ELISA、ECL/MSD等判断


    text = normalize_moa_key(value)
    detected = []

    if any(term in text for term in [
        "electrochem",
        "ecl",
        "meso scale",
        "msd",
    ]):
        detected.append("ECL/MSD")

    if any(term in text for term in [
        "elisa",
        "enzyme-linked",
        "enzyme linked",
    ]):
        detected.append("ELISA")

    if any(term in text for term in [
        "biacore",
        "surface plasmon",
    ]):
        detected.append("SPR/Biacore")

    if "radioimmuno" in text:
        detected.append("Radioimmunoassay")

    if any(term in text for term in [
        "cell-based",
        "cell based",
        "bioassay",
    ]):
        detected.append("Cell-based")

    detected = list(dict.fromkeys(detected))

    if len(detected) == 1:
        return detected[0]

    if len(detected) > 1:
        return "Multiple platforms"

    if any(term in text for term in [
        "bridging",
        "immunoassay",
        "binding assay",
        "anti-drug antibody assay",
    ]):
        return "Other ligand-binding assay"

    return "Platform unclear"


df["ada_assay_platform"] = (
    df["ada_assay"]
    .apply(classify_assay_platform)
)

display(
    df["ada_assay_platform"]
    .value_counts(dropna=False)
)

## 6. 论文图表函数

`ADA ≥10%` 比例图同时输出：

- cohort-level 记录数；
- 独立药物数；
- 独立试验数；
- Wilson 95% 置信区间；
- 对应 Excel 汇总表。

In [ ]:
def safe_filename(name):
    return re.sub(
        r"[^A-Za-z0-9_\-]+",
        "_",
        str(name)
    ).strip("_")


def save_distribution_plot(
    data,
    column,
    output_dir,
    top_n=20,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    counts = (
        data[column]
        .astype("string")
        .fillna("Missing")
        .value_counts()
    )

    if len(counts) > top_n:
        shown = counts.head(top_n).copy()
        shown.loc["Other categories"] = (
            counts.iloc[top_n:].sum()
        )
        counts = shown

    counts = counts.sort_values()

    plt.figure(
        figsize=(9, max(4.5, len(counts) * 0.4))
    )

    plt.barh(
        counts.index.astype(str),
        counts.values
    )

    plt.xlabel("Number of records")
    plt.ylabel(column)
    plt.title(f"Distribution of {column}")
    plt.tight_layout()

    path = (
        output_dir
        / f"{safe_filename(column)}_distribution.png"
    )

    plt.savefig(
        path,
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
    plt.close()

    return path


def wilson_interval(successes, total, z=1.96):
    if total == 0:
        return np.nan, np.nan

    p = successes / total
    denominator = 1 + z**2 / total

    centre = (
        p + z**2 / (2 * total)
    ) / denominator

    margin = (
        z * np.sqrt(
            p * (1 - p) / total
            + z**2 / (4 * total**2)
        ) / denominator
    )

    return centre - margin, centre + margin


def summarize_categorical_feature(
    data,
    column,
):
    summary = (
        data.groupby(column, dropna=False)
        .agg(
            n_records=("ada_high_10", "count"),
            n_high_ADA=("ada_high_10", "sum"),
            high_ADA_rate=("ada_high_10", "mean"),
            n_therapeutics=("therapeutic_id", "nunique"),
            n_trials=("trial_id", "nunique"),
        )
        .reset_index()
    )

    intervals = summary.apply(
        lambda row: wilson_interval(
            row["n_high_ADA"],
            row["n_records"],
        ),
        axis=1
    )

    summary["high_ADA_percent"] = (
        summary["high_ADA_rate"] * 100
    )

    summary["ci_lower_percent"] = [
        lower * 100
        for lower, upper in intervals
    ]

    summary["ci_upper_percent"] = [
        upper * 100
        for lower, upper in intervals
    ]

    return summary.sort_values(
        "n_records",
        ascending=False
    )


def save_ADA_rate_plot(
    data,
    column,
    output_dir,
    minimum_n=10,
    top_n=20,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    summary = summarize_categorical_feature(
        data,
        column
    )

    plot_data = summary.loc[
        summary["n_records"] >= minimum_n
    ].copy()

    if plot_data.empty:
        return None

    plot_data = (
        plot_data
        .sort_values("n_records", ascending=False)
        .head(top_n)
        .sort_values("high_ADA_percent")
    )

    labels = [
        f"{category} (records={records}, drugs={drugs})"
        for category, records, drugs in zip(
            plot_data[column],
            plot_data["n_records"],
            plot_data["n_therapeutics"],
        )
    ]

    # 浮点误差可能产生极小负数，绘图前截断为0
    lower_error = np.maximum(
        plot_data["high_ADA_percent"]
        - plot_data["ci_lower_percent"],
        0
    )

    upper_error = np.maximum(
        plot_data["ci_upper_percent"]
        - plot_data["high_ADA_percent"],
        0
    )

    y = np.arange(len(plot_data))

    plt.figure(
        figsize=(10, max(5, len(plot_data) * 0.5))
    )

    plt.barh(
        y,
        plot_data["high_ADA_percent"]
    )

    plt.errorbar(
        plot_data["high_ADA_percent"],
        y,
        xerr=[lower_error, upper_error],
        fmt="none",
        capsize=3
    )

    plt.yticks(y, labels)
    plt.xlabel("Records with ADA ≥10% (%)")
    plt.ylabel(column)
    plt.title(f"{column} vs ADA ≥10%")
    plt.xlim(0, 100)
    plt.tight_layout()

    path = (
        output_dir
        / f"{safe_filename(column)}_vs_ADA_CI.png"
    )

    plt.savefig(
        path,
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
    plt.close()

    summary.to_excel(
        output_dir
        / f"{safe_filename(column)}_summary.xlsx",
        index=False
    )

    return path

## 7. 生成主文关键图

主文建议使用：

1. ADA frequency 分布；
2. ADA≥10% 类别分布；
3. target group 记录数；
4. target group 与 ADA≥10%（95% CI）；
5. MOA group 记录数；
6. MOA group 与 ADA≥10%（95% CI）。

In [ ]:
for column in [
    "target_group",
    "moa_group",
]:
    save_distribution_plot(
        df,
        column=column,
        output_dir=MAIN_PLOT_DIR,
        top_n=20
    )

    save_ADA_rate_plot(
        df,
        column=column,
        output_dir=MAIN_PLOT_DIR,
        minimum_n=10,
        top_n=20
    )

## 8. 生成补充材料图

这些变量适合放入补充材料，而不是全部塞入主文。

In [ ]:
SUPPLEMENTARY_FEATURES = [
    "species",
    "antibody_backbone_clean",
    "light_chain_clean",
    "fc_modifications_clean",
    "conjugate_modification_clean",
    "expression_system",
    "route_clean",
    "disease_category_clean",
    "ada_assay_platform",
]

for column in SUPPLEMENTARY_FEATURES:
    if column not in df.columns:
        continue

    save_ADA_rate_plot(
        df,
        column=column,
        output_dir=SUPPLEMENT_PLOT_DIR,
        minimum_n=10,
        top_n=20
    )

## 9. 保存清洗标注后的数据与映射审计

输出文件：

- `IDC_modeling_table_cleaned_annotated.xlsx`
- `target_mapping_audit.xlsx`
- `moa_mapping_audit.xlsx`
- 主图、补充图及对应统计表。

In [ ]:
# 靶点映射审计
target_mapping_audit = (
    df[
        [
            "molecule_inn_name_original",
            "molecule_inn_name",
            "target",
            "target_clean",
            "target_group",
            "target_correction_flag",
            "target_correction_note",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "target_group",
            "target_clean",
            "molecule_inn_name",
        ]
    )
)

target_mapping_audit.to_excel(
    REVIEW_DIR / "target_mapping_audit.xlsx",
    index=False
)


# MOA 映射审计
moa_mapping_audit = (
    df[
        [
            "molecule_inn_name_original",
            "molecule_inn_name",
            "target_clean",
            "mechanism_of_action",
            "mechanism_of_action_reviewed",
            "moa_group",
            "moa_classification_source",
            "moa_review_flag",
            "moa_review_note",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "moa_group",
            "molecule_inn_name",
        ]
    )
)

moa_mapping_audit.to_excel(
    REVIEW_DIR / "moa_mapping_audit.xlsx",
    index=False
)


# 完整清洗标注表
cleaned_path = (
    OUTPUT_DIR
    / "IDC_modeling_table_cleaned_annotated.xlsx"
)

df.to_excel(
    cleaned_path,
    index=False
)

print("清洗标注表：", cleaned_path.resolve())
print("主图目录：", MAIN_PLOT_DIR.resolve())
print("补充图目录：", SUPPLEMENT_PLOT_DIR.resolve())
print("复核表目录：", REVIEW_DIR.resolve())

# 建模就绪文件：删除目标泄漏列，同时保留排除记录和质量控制表。
model_ready_path = OUTPUT_DIR / "IDC_modeling_table_final_model_ready.xlsx"
model_qc = pd.DataFrame([
    {"check": "final_modeling_rows", "value": len(modeling_df), "status": "PASS"},
    {"check": "duplicate_id_rows", "value": int(modeling_df["idc_row_id"].duplicated(False).sum()), "status": "PASS"},
    {"check": "missing_required_values", "value": int(modeling_df[required_model_fields].isna().sum().sum()), "status": "PASS"},
    {"check": "count_vs_frequency_review_flags", "value": int(modeling_df["ada_count_consistency_status"].eq("REVIEW_COUNT_MISMATCH").sum()), "status": "CAUTION"},
    {"check": "binomial_count_model_eligible", "value": int(modeling_df["binomial_count_model_eligible"].sum()), "status": "INFO"},
])
with pd.ExcelWriter(model_ready_path) as writer:
    modeling_df.to_excel(writer, sheet_name="Modeling_Data", index=False)
    excluded_duplicate_rows.to_excel(writer, sheet_name="Excluded_Duplicates", index=False)
    model_qc.to_excel(writer, sheet_name="QC_Summary", index=False)

print("建模就绪文件：", model_ready_path.resolve())
